# 🚀 MiniMax H3 (Hailuo 3.0) 纯本地 Colab 视频生产工作流

> **核心原则**：
> 1. **纯本地运行**：运行开源剪枝与量化权重，绝不调用外部商业 API。
> 2. **硬件门禁与三重防扣费**：强制要求算力 ≥ 8.0（首选 **L4 24GB**）。遇 T4 自动阻断并关机；配备后台 Watchdog 超时熔断守门员。
> 3. **无脸视频批产流**：提示词融合环境声效与视觉构图，产物直存 Google Drive。


### 步骤 1：挂载 Google Drive 与启动算力单元看门狗（Watchdog）
- 自动挂载 Drive 以便持久化保存成片和模型缓存。
- **看门狗安全线程**：若执行时间超过设定时限（默认 2.5 小时），强制触发关机，彻底防止遗忘网页导致 200 算力单元耗尽。


In [1]:
from google.colab import drive, runtime
import os, time, threading

# 官方推荐幂等挂载检测，防止重复点击授权或重挂载报错
if not os.path.exists("/content/drive/MyDrive"):
    print("⏳ 正在通过官方接口挂载 Google Drive（请在弹出框中点击授权）...")
    drive.mount("/content/drive")
else:
    print("✅ Google Drive 处于已挂载就绪状态")

os.makedirs('/content/drive/MyDrive/MiniMax_Outputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MiniMax_Models', exist_ok=True)
print("📁 云端模型持久化缓存: /content/drive/MyDrive/MiniMax_Models")
print("🎬 成片与日志最终落盘: /content/drive/MyDrive/MiniMax_Outputs")

# 算力看门狗守护线程 (防闲置偷跑算力)
WATCHDOG_HOURS = 2.5 # 设置单次最大运行小时上限
def watchdog():
    time.sleep(int(WATCHDOG_HOURS * 3600))
    print("\n🚨 [WATCHDOG 熔断] 运行时已超最大限制，正在自动销毁实例以保卫算力！")
    try:
        os.sync()
        runtime.unassign()
    except Exception:
        os.kill(1, 15)

threading.Thread(target=watchdog, daemon=True).start()
print(f"🛡️  算力看门狗已激活：{WATCHDOG_HOURS} 小时未退出将强制终止实例。")


⏳ 正在通过官方接口挂载 Google Drive（请在弹出框中点击授权）...
Mounted at /content/drive
📁 云端模型持久化缓存: /content/drive/MyDrive/MiniMax_Models
🎬 成片与日志最终落盘: /content/drive/MyDrive/MiniMax_Outputs
🛡️  算力看门狗已激活：2.5 小时未退出将强制终止实例。


### 步骤 2：硬件环境门禁检测（阻断 T4，验证 L4/A100）
- 若检测到 T4（算力 7.5），会自动触发关机拦截，保护额度。


In [2]:
import torch, shutil
from google.colab import runtime

if not torch.cuda.is_available():
    print("❌ 未检测到 GPU！")
    runtime.unassign()
    raise RuntimeError("请在菜单：代码执行程序 -> 更改运行时类型 中分配 GPU (首选 L4)。")

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
cc = major + minor / 10.0
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
disk_free_gb = shutil.disk_usage('/content').free / (1024 ** 3)

print(f"🖥️ 当前显卡: {gpu_name} (算力 {cc:.1f})")
print(f"💾 显存容量: {vram_gb:.2f} GB | 💽 可用磁盘: {disk_free_gb:.1f} GB")

if cc < 8.0:
    print(f"\n🚨 致命阻断：{gpu_name} 算力 {cc:.1f} < 8.0，不支持 Int8 ConvRot 算子！")
    print("🛑 正在自动销毁实例，防止白白消耗算力单元...")
    runtime.unassign()
    raise RuntimeError(f"硬件不达标 ({gpu_name})，已自动断开。请重新分配 L4 实例。")

print("\n🎉 硬件门禁通过（≥ 8.0），允许执行！")


🖥️ 当前显卡: NVIDIA L4 (算力 8.9)
💾 显存容量: 22.03 GB | 💽 可用磁盘: 235.6 GB

🎉 硬件门禁通过（≥ 8.0），允许执行！


### 步骤 3：获取项目脚本与安装 ComfyUI 核心运行环境
- 自动准备 ComfyUI 运行环境。
- 具备完全自包含兜底：若未联网克隆仓库，自动嵌入标准生产工作流与批处理模块，确保单文件上传也能 100% 运行。


In [3]:
%cd /content

# 1. 安装最新版 ComfyUI
if not os.path.exists('/content/ComfyUI'):
    print("📦 正在安装 ComfyUI 核心环境...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q huggingface_hub accelerate safetensors
    %cd /content

# 2. 准备项目工作空间 (/content/minimax-h3-colab-workflow)
import os, sys, json
from pathlib import Path

REPO_DIR = Path("/content/minimax-h3-colab-workflow")
if not (REPO_DIR / "scripts" / "batch_runner.py").exists():
    print("📦 尝试克隆项目仓库...")
    !git clone https://github.com/kuhung/minimax-h3-colab-workflow.git /content/minimax-h3-colab-workflow 2>/dev/null || true

# 3. 自包含兜底初始化：即使仅单文件上传，也能自动写入内置的标准工作流与脚本
(REPO_DIR / "workflows").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "configs").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "scripts").mkdir(parents=True, exist_ok=True)

embedded_wf_pdd8 = '{\n  "1": {\n    "inputs": {\n      "unet_name": "minimax_h3_fl2va_pruned_int8_convrot.safetensors",\n      "weight_dtype": "default"\n    },\n    "class_type": "UNETLoader",\n    "_meta": {\n      "title": "MiniMax H3 Diffusion Model (Pruned Int8)"\n    }\n  },\n  "2": {\n    "inputs": {\n      "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n      "type": "minimax"\n    },\n    "class_type": "CLIPLoader",\n    "_meta": {\n      "title": "MiniMax H3 Text Encoder (NVFP4)"\n    }\n  },\n  "4": {\n    "inputs": {\n      "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"\n    },\n    "class_type": "VAELoader",\n    "_meta": {\n      "title": "MiniMax H3 Video VAE"\n    }\n  },\n  "10": {\n    "inputs": {\n      "lora_name": "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",\n      "strength_model": 1.0,\n      "model": [\n        "1",\n        0\n      ]\n    },\n    "class_type": "LoraLoaderModelOnly",\n    "_meta": {\n      "title": "PDD 8-Step Distillation LoRA"\n    }\n  },\n  "5": {\n    "inputs": {\n      "width": 960,\n      "height": 544,\n      "length": 124\n    },\n    "class_type": "EmptyMiniMaxH3LatentAV",\n    "_meta": {\n      "title": "Empty MiniMax H3 Latent"\n    }\n  },\n  "6": {\n    "inputs": {\n      "text": "A close-up macro shot of dark roast espresso dripping slowly from a bottomless portafilter into a clear glass. Golden-brown crema forming on top with rich swirls. Soft ambient morning light from the side. Sound of rhythmic espresso drops falling into liquid, subtle steam hiss, quiet morning acoustic ambiance.",\n      "clip": [\n        "2",\n        0\n      ]\n    },\n    "class_type": "CLIPTextEncode",\n    "_meta": {\n      "title": "Positive Prompt"\n    }\n  },\n  "7": {\n    "inputs": {\n      "text": "watermark, text, blur, low quality, deformed, jerky motion, noisy static",\n      "clip": [\n        "2",\n        0\n      ]\n    },\n    "class_type": "CLIPTextEncode",\n    "_meta": {\n      "title": "Negative Prompt"\n    }\n  },\n  "3": {\n    "inputs": {\n      "seed": 42,\n      "steps": 8,\n      "cfg": 3.0,\n      "sampler_name": "euler",\n      "scheduler": "simple",\n      "denoise": 1.0,\n      "model": [\n        "10",\n        0\n      ],\n      "positive": [\n        "6",\n        0\n      ],\n      "negative": [\n        "7",\n        0\n      ],\n      "latent_image": [\n        "5",\n        0\n      ]\n    },\n    "class_type": "KSampler",\n    "_meta": {\n      "title": "KSampler (PDD 8-Steps Turbo Production)"\n    }\n  },\n  "8": {\n    "inputs": {\n      "samples": [\n        "3",\n        0\n      ],\n      "vae": [\n        "4",\n        0\n      ]\n    },\n    "class_type": "VAEDecode",\n    "_meta": {\n      "title": "VAE Decode"\n    }\n  },\n  "9": {\n    "inputs": {\n      "images": [\n        "8",\n        0\n      ],\n      "filename_prefix": "H3_PDD8_Production",\n      "fps": 24.0,\n      "lossless": false,\n      "quality": 85,\n      "method": "default"\n    },\n    "class_type": "SaveAnimatedWEBP",\n    "_meta": {\n      "title": "Save Production Video"\n    }\n  }\n}'

embedded_prompts = '[\n  {\n    "id": "shot_01_coffee_macro",\n    "title": "咖啡机微距流速",\n    "prompt": "A close-up macro shot of dark roast espresso dripping slowly from a bottomless portafilter into a clear glass. Golden-brown crema forming on top with rich swirls. Soft ambient morning light from the side. Sound of rhythmic espresso drops falling into liquid, subtle steam hiss, quiet morning acoustic ambiance.",\n    "negative_prompt": "watermark, text, blur, low quality, deformed, jerky motion, noisy static",\n    "width": 960,\n    "height": 544,\n    "length": 124,\n    "steps": 8,\n    "seed": 42\n  },\n  {\n    "id": "shot_02_server_datacenter",\n    "title": "数据中心矩阵光影",\n    "prompt": "Cinematic slow forward dolly tracking through an endless corridor of high-density AI server racks. Blue and amber LED activity lights flickering rhythmically. Volumetric cold mist cooling effects. Deep industrial hum of ventilation fans, intermittent electronic beep echoes in an enclosed server hall.",\n    "negative_prompt": "watermark, text, blur, low quality, deformed, shaky, glitchy",\n    "width": 1344,\n    "height": 768,\n    "length": 124,\n    "steps": 8,\n    "seed": 108\n  },\n  {\n    "id": "shot_03_typing_hands",\n    "title": "极客机械键盘打字",\n    "prompt": "Side perspective close-up shot of anonymous hands typing rapidly on a custom mechanical keyboard with tactile switches. Dark desk mat with warm retro desk lamp lighting. In the background, soft bokeh of terminal text editor. Crisp mechanical switch clacks with satisfying acoustic resonance, subtle cloth rustle of sleeves.",\n    "negative_prompt": "watermark, text, deformed fingers, extra fingers, blur, low quality, noisy audio",\n    "width": 1344,\n    "height": 768,\n    "length": 124,\n    "steps": 8,\n    "seed": 2048\n  }\n]\n'

embedded_batch_runner = '#!/usr/bin/env python3\n"""\nHeadless Batch Runner for MiniMax H3 on ComfyUI (2.0 Architecture).\nFeatures:\n1. Native 2.0 Topology Support: MiniMaxH3ImageToVideo + BasicGuider + SamplerCustomAdvanced (CFG=1.0).\n2. Backward compatibility with legacy KSampler graphs.\n3. 17k + 5 Frame Alignment (align_frame_count).\n4. Structured Multi-shot / Multimodal Prompt Assembler (format_h3_prompt).\n5. Dynamic /object_info Contract Validation & Unsupported Parameter Pruning.\n6. Timeout Interrupt & Direct Google Drive Export.\n"""\n\nimport os\nimport sys\nimport copy\nimport json\nimport time\nimport shutil\nimport urllib.request\nimport urllib.parse\nimport urllib.error\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\ndef align_frame_count(n: int) -> int:\n    """Align frame count upwards to 17k + 5 (MiniMax H3 latent temporal grid at 24fps)."""\n    n = max(5, int(n))\n    return n + (5 - n % 17) % 17\n\ndef format_h3_prompt(task: dict) -> str:\n    """\n    Format MiniMax H3 structured prompt according to official multi-shot / multimodal spec.\n    If task contains \'shots\' or \'overall_soundscape\', formats:\n      integrated_multimodal_description:\n      [Shot 1] ...\n      [Shot 2] At MM:SS.mmm, ...\n\n      overall_soundscape: ...\n\n      non_diegetic_music: ...\n    If task already provides plain \'prompt\', returns it directly.\n    """\n    if not isinstance(task, dict):\n        return str(task)\n\n    shots = task.get("shots")\n    soundscape = task.get("overall_soundscape")\n    music = task.get("non_diegetic_music")\n\n    if not shots and not soundscape and not music:\n        return task.get("prompt", "")\n\n    parts = []\n    if shots:\n        shot_lines = []\n        for i, s in enumerate(shots, 1):\n            if isinstance(s, dict):\n                t_str = s.get("time")\n                desc = s.get("description", s.get("prompt", ""))\n                if i == 1 and (not t_str or t_str == "00:00.000" or t_str == "0"):\n                    shot_lines.append(f"[Shot 1] {desc}")\n                else:\n                    t_val = t_str if t_str else f"00:{((i-1)*5):02d}.000"\n                    shot_lines.append(f"[Shot {i}] At {t_val}, {desc}")\n            else:\n                if i == 1:\n                    shot_lines.append(f"[Shot 1] {s}")\n                else:\n                    shot_lines.append(f"[Shot {i}] At 00:{((i-1)*5):02d}.000, {s}")\n        parts.append("integrated_multimodal_description:\\n" + "\\n".join(shot_lines))\n    elif task.get("prompt"):\n        parts.append(f"integrated_multimodal_description:\\n{task[\'prompt\']}")\n\n    if soundscape:\n        parts.append(f"overall_soundscape: {soundscape}")\n    if music:\n        parts.append(f"non_diegetic_music: {music}")\n\n    return "\\n\\n".join(parts)\n\ndef fetch_object_info(server_url: str) -> Optional[dict]:\n    """Queries ComfyUI /object_info endpoint."""\n    try:\n        req = urllib.request.Request(f"{server_url}/object_info")\n        with urllib.request.urlopen(req, timeout=10) as response:\n            return json.loads(response.read().decode("utf-8"))\n    except Exception as e:\n        print(f"⚠️ 获取 ComfyUI /object_info 提示: {e}")\n        return None\n\ndef validate_and_adapt_workflow(wf: dict, obj_info: Optional[dict]) -> dict:\n    """\n    Validates node classes and input fields against ComfyUI /object_info.\n    Automatically prunes unsupported optional inputs to prevent ComfyUI 400 rejection.\n    """\n    if not obj_info:\n        return wf\n\n    cleaned_wf = copy.deepcopy(wf)\n    for nid, node in cleaned_wf.items():\n        if not isinstance(node, dict):\n            continue\n        ctype = node.get("class_type")\n        if not ctype:\n            continue\n        if ctype not in obj_info:\n            raise ValueError(f"节点 [{nid}] 类名 \'{ctype}\' 未在 ComfyUI 中注册！请检查扩展是否安装完整。")\n\n        node_spec = obj_info[ctype].get("input", {})\n        req = set(node_spec.get("required", {}).keys())\n        opt = set(node_spec.get("optional", {}).keys())\n        valid_keys = req | opt\n        inputs = node.get("inputs", {})\n\n        to_remove = [k for k in list(inputs.keys()) if k not in valid_keys]\n        for k in to_remove:\n            print(f"⚠️ 节点 [{nid}: {ctype}] 移除非当前版本支持的输入参数: \'{k}\'")\n            del inputs[k]\n\n        missing_req = [k for k in req if k not in inputs]\n        if missing_req:\n            raise ValueError(f"节点 [{nid}: {ctype}] 缺失必要输入参数: {missing_req}")\n\n    return cleaned_wf\n\ndef post_prompt(server_url: str, prompt_workflow: dict) -> Optional[str]:\n    """Submits workflow JSON to ComfyUI API /prompt."""\n    data = json.dumps({"prompt": prompt_workflow, "client_id": "batch_runner_2.0"}).encode("utf-8")\n    req = urllib.request.Request(f"{server_url}/prompt", data=data, headers={"Content-Type": "application/json"})\n    try:\n        with urllib.request.urlopen(req, timeout=15) as response:\n            res = json.loads(response.read().decode("utf-8"))\n            if res.get("error") or res.get("node_errors"):\n                print(f"❌ ComfyUI 拒绝执行工作流: {res}")\n                return None\n            return res.get("prompt_id")\n    except urllib.error.HTTPError as e:\n        err_body = e.read().decode("utf-8", errors="replace")\n        print(f"❌ 请求 ComfyUI API 失败 [HTTP {e.code}]: {err_body}")\n        return None\n    except Exception as e:\n        print(f"❌ 请求 ComfyUI API 失败: {e}")\n        return None\n\ndef interrupt_comfyui(server_url: str):\n    """Sends /interrupt POST to halt runaway or timed out jobs and free VRAM."""\n    try:\n        req = urllib.request.Request(f"{server_url}/interrupt", data=b"{}", headers={"Content-Type": "application/json"})\n        with urllib.request.urlopen(req, timeout=5) as response:\n            if response.status in (200, 204):\n                print("🛑 已成功下发 ComfyUI 中断指令 (/interrupt)，停止去噪并释放显存。")\n    except Exception as e:\n        print(f"⚠️ 下发 ComfyUI 中断请求失败: {e}")\n\ndef wait_for_completion(server_url: str, prompt_id: str, timeout_sec: int = 900) -> Optional[dict]:\n    """Polls ComfyUI /history/{prompt_id} until generation finishes or times out."""\n    start_time = time.time()\n    last_print = 0\n    while time.time() - start_time < timeout_sec:\n        try:\n            req = urllib.request.Request(f"{server_url}/history/{prompt_id}")\n            with urllib.request.urlopen(req, timeout=5) as response:\n                history = json.loads(response.read().decode("utf-8"))\n                if prompt_id in history:\n                    return history[prompt_id]\n        except Exception:\n            pass\n\n        elapsed = int(time.time() - start_time)\n        if elapsed - last_print >= 20:\n            print(f"   ⏳ 渲染去噪中... 已耗时 {elapsed}s")\n            last_print = elapsed\n\n        time.sleep(3)\n\n    print(f"❌ 任务超时 ({timeout_sec}s)，正在中断当前生成...")\n    interrupt_comfyui(server_url)\n    return None\n\ndef extract_output_files(history_data: dict, comfy_output_dir: Path) -> List[Path]:\n    """Extracts actual output files from ComfyUI history outputs dict."""\n    files = []\n    outputs = history_data.get("outputs", {})\n    for node_id, node_data in outputs.items():\n        for category in ["videos", "gifs", "images", "audio"]:\n            for item in node_data.get(category, []):\n                filename = item.get("filename")\n                if filename:\n                    subfolder = item.get("subfolder", "")\n                    target_file = comfy_output_dir / subfolder / filename\n                    if target_file.exists():\n                        files.append(target_file)\n    return files\n\ndef run_batch(\n    workflow_path: str,\n    prompts_path: Optional[str] = None,\n    output_dir: str = "/content/drive/MyDrive/MiniMax_Outputs",\n    server_url: str = "http://127.0.0.1:8188",\n    comfy_output_dir: str = "/content/ComfyUI/output",\n    timeout_per_shot: int = 900,\n    tasks_list: Optional[List[dict]] = None,\n    active_models: Optional[Dict[str, str]] = None,\n):\n    print("=" * 68)\n    print("🎬 MiniMax H3 批处理队列执行器 (Batch Runner 2.0 - BasicGuider / No-CFG)")\n    print(f"📄 工作流模板: {workflow_path}")\n\n    out_path = Path(output_dir)\n    out_path.mkdir(parents=True, exist_ok=True)\n    log_file = out_path / "render_history.jsonl"\n    comfy_out_path = Path(comfy_output_dir)\n\n    with open(workflow_path, "r", encoding="utf-8") as f:\n        base_workflow = json.load(f)\n\n    if tasks_list is not None:\n        tasks = tasks_list\n        print(f"📋 任务清单: 内存任务列表 ({len(tasks)} 条)")\n    elif prompts_path:\n        print(f"📋 任务清单: {prompts_path}")\n        with open(prompts_path, "r", encoding="utf-8") as f:\n            tasks = json.load(f)\n    else:\n        raise ValueError("必须提供 prompts_path 或 tasks_list 其中之一！")\n\n    # 动态解析当前环境实际权重（优先使用传入的 active_models，未传时自动探测 models 目录）\n    resolved_models = dict(active_models) if active_models else {}\n    if not resolved_models and comfy_out_path.parent.exists():\n        models_dir = comfy_out_path.parent / "models"\n        diff_dir = models_dir / "diffusion_models"\n        vae_dir = models_dir / "vae"\n        lora_dir = models_dir / "loras"\n        if diff_dir.exists():\n            for cand in ["minimax_h3_fl2va_pruned_fp8_scaled.safetensors", "minimax_h3_fl2va_pruned_int8_convrot.safetensors"]:\n                if (diff_dir / cand).exists():\n                    resolved_models["diffusion_model"] = cand\n                    break\n        if vae_dir.exists():\n            for cand in ["minimax_h3_video_vae_fp16.safetensors", "minimax_h3_video_vae_int8_convrot.safetensors"]:\n                if (vae_dir / cand).exists():\n                    resolved_models["video_vae"] = cand\n                    break\n        if lora_dir.exists():\n            for cand in ["minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"]:\n                if (lora_dir / cand).exists():\n                    resolved_models["lora"] = cand\n                    break\n\n    if resolved_models:\n        print(f"🎯 动态绑定的活动权重: {resolved_models}")\n\n    print(f"💾 最终输出落盘路径: {output_dir}")\n    print("=" * 68)\n    print(f"📊 共检测到 {len(tasks)} 个排产镜头任务\\n")\n\n    # 预检 /object_info 获取节点契约\n    obj_info = fetch_object_info(server_url)\n\n    for idx, task in enumerate(tasks, 1):\n        task_id = task.get("id", f"shot_{idx:02d}")\n        title = task.get("title", task_id)\n        prompt_text = format_h3_prompt(task)\n        neg_text = task.get("negative_prompt", "")\n        steps = int(task.get("steps", 8))\n        seed = int(task.get("seed", int(time.time())))\n        width = int(task.get("width", 1344))\n        height = int(task.get("height", 768))\n        raw_length = int(task.get("length", 124))\n        length = align_frame_count(raw_length)\n        if length != raw_length:\n            print(f"   ℹ️ 帧数自动网格对齐 (17k + 5): {raw_length} -> {length}")\n\n        print(f"[{idx}/{len(tasks)}] 正在启动: {task_id} - {title}")\n        print(f"   📐 规格: {width}x{height} | {length} 帧 | 步数 {steps} | 种子 {seed}")\n        print(f"   📝 提示词: {prompt_text[:90]}...")\n\n        # 克隆工作流并动态注入参数\n        wf = copy.deepcopy(base_workflow)\n\n        # 优先解析采样器图拓扑连接，准确定位正/负提示词与 Latent 节点 ID（不单纯依赖 _meta 标题）\n        pos_node_ids = set()\n        neg_node_ids = set()\n        latent_node_ids = set()\n\n        for nid, n in wf.items():\n            if not isinstance(n, dict):\n                continue\n            ctype = n.get("class_type")\n            inps = n.get("inputs", {})\n            if ctype in ("KSampler", "KSamplerAdvanced"):\n                if isinstance(inps.get("positive"), list) and len(inps["positive"]) > 0:\n                    pos_node_ids.add(str(inps["positive"][0]))\n                if isinstance(inps.get("negative"), list) and len(inps["negative"]) > 0:\n                    neg_node_ids.add(str(inps["negative"][0]))\n                if isinstance(inps.get("latent_image"), list) and len(inps["latent_image"]) > 0:\n                    latent_node_ids.add(str(inps["latent_image"][0]))\n            elif ctype == "BasicGuider":\n                if isinstance(inps.get("conditioning"), list) and len(inps["conditioning"]) > 0:\n                    pos_node_ids.add(str(inps["conditioning"][0]))\n\n        # 遍历节点并匹配注入\n        for node_id, node in wf.items():\n            if not isinstance(node, dict):\n                continue\n            class_type = node.get("class_type")\n            inputs = node.get("inputs", {})\n            title_meta = node.get("_meta", {}).get("title", "")\n\n            # 动态权重注入\n            if class_type == "UNETLoader" and "diffusion_model" in resolved_models:\n                inputs["unet_name"] = resolved_models["diffusion_model"]\n            elif class_type == "VAELoader" and "video_vae" in resolved_models:\n                if inputs.get("vae_name") != "minimax_h3_audio_vae_fp32.safetensors":\n                    inputs["vae_name"] = resolved_models["video_vae"]\n            elif class_type in ("LoraLoaderModelOnly", "LoraLoader") and "lora" in resolved_models:\n                inputs["lora_name"] = resolved_models["lora"]\n\n            # 2.0 新节点注入\n            elif class_type == "MiniMaxH3ImageToVideo":\n                inputs["prompt"] = prompt_text\n                inputs["width"] = width\n                inputs["height"] = height\n                inputs["length"] = length\n            elif class_type == "RandomNoise":\n                inputs["noise_seed"] = seed\n            elif class_type == "KSamplerSelect":\n                if "sampler_name" in task:\n                    inputs["sampler_name"] = task["sampler_name"]\n            elif class_type == "BasicScheduler":\n                inputs["steps"] = steps\n                if "scheduler" in task:\n                    inputs["scheduler"] = task["scheduler"]\n                if "denoise" in task:\n                    inputs["denoise"] = float(task["denoise"])\n\n            # 输出节点注入\n            elif class_type in ("SaveVideo", "SaveAnimatedWEBP", "VHS_VideoCombine"):\n                inputs["filename_prefix"] = f"{task_id}_{seed}"\n                if class_type == "SaveVideo":\n                    inputs.setdefault("format", "mp4")\n                elif class_type == "SaveAnimatedWEBP":\n                    inputs.setdefault("fps", 24.0)\n                    inputs.setdefault("lossless", False)\n                    inputs.setdefault("quality", 85)\n                    inputs.setdefault("method", "default")\n            elif class_type == "CreateVideo":\n                inputs.setdefault("fps", 24.0)\n\n            # 向后兼容经典 1.0 节点\n            elif class_type == "CLIPTextEncode":\n                t_lower = title_meta.lower()\n                is_pos = (node_id in pos_node_ids) or ("positive" in t_lower) or ("正向" in title_meta)\n                is_neg = (node_id in neg_node_ids) or ("negative" in t_lower) or ("负向" in title_meta)\n                if is_pos and not is_neg:\n                    inputs["text"] = prompt_text\n                    print(f"   📝 [节点 {node_id}] 正向提示词注入成功: {prompt_text[:50]}...")\n                elif is_neg and not is_pos:\n                    if neg_text:\n                        inputs["text"] = neg_text\n                        print(f"   🛡️ [节点 {node_id}] 负向提示词注入成功: {neg_text[:40]}...")\n                elif not pos_node_ids and not neg_node_ids:\n                    inputs["text"] = prompt_text\n                    print(f"   📝 [节点 {node_id}] 独立提示词注入成功: {prompt_text[:50]}...")\n\n            elif class_type in ("EmptyMiniMaxH3LatentAV", "EmptyH3LatentVideo", "EmptyLatentVideo") or (node_id in latent_node_ids):\n                inputs["width"] = width\n                inputs["height"] = height\n                inputs["length"] = length\n                print(f"   📐 [节点 {node_id}: {class_type}] 规格尺寸注入: {width}x{height}, {length} 帧")\n\n            elif class_type in ("KSampler", "KSamplerAdvanced"):\n                inputs["steps"] = steps\n                inputs["seed"] = seed\n                if "cfg" in task:\n                    inputs["cfg"] = float(task["cfg"])\n\n        # 校验并适配 ComfyUI 节点输入契约\n        try:\n            wf = validate_and_adapt_workflow(wf, obj_info)\n        except Exception as ve:\n            print(f"❌ 契约校验失败: {ve}")\n            continue\n\n        # 提交至 ComfyUI\n        t_start = time.time()\n        prompt_id = post_prompt(server_url, wf)\n        if not prompt_id:\n            print(f"❌ 提交任务 {task_id} 失败，跳过并记录。")\n            log_record = {\n                "timestamp": datetime.now().isoformat(),\n                "task_id": task_id,\n                "status": "submission_failed",\n                "title": title,\n                "cost_seconds": 0,\n                "prompt": prompt_text,\n                "saved_files": []\n            }\n            with open(log_file, "a", encoding="utf-8") as lf:\n                lf.write(json.dumps(log_record, ensure_ascii=False) + "\\n")\n            continue\n\n        print(f"   🚀 已推入执行队列 (Prompt ID: {prompt_id})")\n        res = wait_for_completion(server_url, prompt_id, timeout_sec=timeout_per_shot)\n        t_cost = round(time.time() - t_start, 2)\n\n        # 状态检查\n        status_info = res.get("status", {}) if res else {}\n        status_str = status_info.get("status_str", "unknown")\n        is_completed = status_info.get("completed", False)\n\n        copied_files = []\n        if res and is_completed and status_str == "success":\n            # 优先从 outputs 解析生成文件\n            found_files = extract_output_files(res, comfy_out_path)\n            # 若为空，兜底 glob 扫描\n            if not found_files and comfy_out_path.exists():\n                found_files = list(comfy_out_path.glob(f"{task_id}_{seed}*"))\n\n            if len(found_files) == 0:\n                log_status = "failed_no_output"\n                print(f"   ❌ 渲染完成但未产生任何输出文件！")\n            else:\n                print(f"   ✅ 渲染成功！耗时: {t_cost}s")\n                for f in found_files:\n                    target_dst = out_path / f.name\n                    shutil.copy2(f, target_dst)\n                    copied_files.append(str(target_dst))\n                    print(f"   💾 成片已同步至云盘: {target_dst.name}")\n                log_status = "success"\n        else:\n            log_status = "failed" if res else "timeout"\n            err_msg = status_info.get("messages", [])\n            print(f"   ❌ 任务状态: {log_status}，耗时: {t_cost}s，原因: {err_msg}")\n\n        # 写入历史日志\n        log_record = {\n            "timestamp": datetime.now().isoformat(),\n            "task_id": task_id,\n            "title": title,\n            "status": log_status,\n            "cost_seconds": t_cost,\n            "resolution": f"{width}x{height}",\n            "length_frames": length,\n            "steps": steps,\n            "seed": seed,\n            "prompt": prompt_text,\n            "saved_files": copied_files\n        }\n        with open(log_file, "a", encoding="utf-8") as lf:\n            lf.write(json.dumps(log_record, ensure_ascii=False) + "\\n")\n\n        print("-" * 50)\n\n    print("\\n" + "=" * 68)\n    print(f"🎉 全部批处理任务执行完毕！")\n    print(f"📂 成片与日志均已持久化至: {out_path.resolve()}")\n    print("=" * 68)\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser(description="ComfyUI Batch Runner for MiniMax H3 (2.0)")\n    parser.add_argument("--workflow", default="workflows/h3_t2v_pdd8_production.json", help="Path to workflow JSON")\n    parser.add_argument("--prompts", default="configs/prompts.example.json", help="Path to prompts JSON")\n    parser.add_argument("--output", default="/content/drive/MyDrive/MiniMax_Outputs", help="Output directory")\n    parser.add_argument("--server", default="http://127.0.0.1:8188", help="ComfyUI server URL")\n    parser.add_argument("--comfy-output", default="/content/ComfyUI/output", help="ComfyUI output dir")\n    parser.add_argument("--timeout", type=int, default=900, help="Max seconds per shot")\n    args = parser.parse_args()\n\n    run_batch(\n        workflow_path=args.workflow,\n        prompts_path=args.prompts,\n        output_dir=args.output,\n        server_url=args.server,\n        comfy_output_dir=args.comfy_output,\n        timeout_per_shot=args.timeout\n    )\n'

(REPO_DIR / "workflows" / "h3_t2v_pdd8_production.json").write_text(embedded_wf_pdd8, encoding="utf-8")
print("✅ 已自包含写入: workflows/h3_t2v_pdd8_production.json")

if not (REPO_DIR / "configs" / "prompts.example.json").exists():
    (REPO_DIR / "configs" / "prompts.example.json").write_text(embedded_prompts, encoding="utf-8")
    print("✅ 已自包含写入: configs/prompts.example.json")

(REPO_DIR / "scripts" / "batch_runner.py").write_text(embedded_batch_runner, encoding="utf-8")
print("✅ 已自包含写入: scripts/batch_runner.py")

print("✅ 项目运行环境与工作流模板准备就绪: /content/minimax-h3-colab-workflow")


/content
📦 正在安装 ComfyUI 核心环境...
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 62810, done.
remote: Counting objects: 100% (1060/1060), done.
remote: Compressing objects: 100% (415/415), done.
remote: Total 62810 (delta 961), reused 649 (delta 645), pack-reused 61750 (from 3)
Receiving objects: 100% (62810/62810), 98.01 MiB | 19.84 MiB/s, done.
Resolving deltas: 100% (43496/43496), done.
/content/ComfyUI
/content
📦 尝试克隆项目仓库...
✅ 已自包含写入: workflows/h3_t2v_pdd8_production.json
✅ 已自包含写入: configs/prompts.example.json
✅ 已自包含写入: scripts/batch_runner.py
✅ 项目运行环境与工作流模板准备就绪: /content/minimax-h3-colab-workflow


### 步骤 4：下载或软链接 MiniMax H3 剪枝权重（校验完整性与 Drive 缓存）
- 优先检索 Drive 缓存；若文件受损自动重新拉取；仅同步完整文件。


In [4]:
import os, shutil, time, struct, json
from pathlib import Path
from huggingface_hub import hf_hub_download

# HuggingFace 官方 Comfy-Org/MiniMax-H3 精确字节数规范
EXACT_SIZES = {
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20970379616,
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15687142551,
    "vae/minimax_h3_video_vae_int8_convrot.safetensors": 2811065184,
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605254808,
    "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors": 1956193000,
}

MODELS_SPEC = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models", EXACT_SIZES["diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors"]),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders", EXACT_SIZES["text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"]),
    ("vae/minimax_h3_video_vae_int8_convrot.safetensors", "vae", EXACT_SIZES["vae/minimax_h3_video_vae_int8_convrot.safetensors"]),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae", EXACT_SIZES["vae/minimax_h3_audio_vae_fp32.safetensors"]),
    ("loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "loras", EXACT_SIZES["loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"]),
]

DRIVE_MODELS = Path("/content/drive/MyDrive/MiniMax_Models")
LOCAL_MODELS = Path("/content/ComfyUI/models")

def check_safetensors_file(file_path: Path, expected_size: int) -> bool:
    """严格校验文件大小与 safetensors 头部元数据边界，防止半截残损文件"""
    if not file_path.is_file():
        return False
    try:
        actual_size = file_path.stat().st_size
        if actual_size != expected_size:
            return False
        with open(file_path, "rb") as f:
            header_bytes = f.read(8)
            if len(header_bytes) < 8:
                return False
            hlen = struct.unpack("<Q", header_bytes)[0]
            header = json.loads(f.read(hlen).decode("utf-8"))
        max_end = max(v["data_offsets"][1] for k, v in header.items() if k != "__metadata__")
        return (8 + hlen + max_end) == actual_size
    except Exception:
        return False

print("🔗 正在挂载并校验 5 大核心模型权重 (本地 NVMe 优先 + HF 直连 + Drive 容灾)...")
t_m0 = time.time()

for rel_path, subfolder, exact_bytes in MODELS_SPEC:
    fname = Path(rel_path).name
    dst_dir = LOCAL_MODELS / subfolder
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_link = dst_dir / fname
    drive_sub_file = DRIVE_MODELS / subfolder / fname
    drive_flat_file = DRIVE_MODELS / fname

    # 1. 检查本地 NVMe 是否已有真实物理文件且通过完整性校验
    if not dst_link.is_symlink() and dst_link.is_file() and check_safetensors_file(dst_link, exact_bytes):
        print(f"   ✅ 本地 NVMe 高速缓存已就绪 (真实物理文件): {subfolder}/{fname} ({exact_bytes / (1024**3):.2f} GiB)")
        continue

    # 若本地是软链接或残损文件，先清除，确保后续获得本地高速物理文件
    if dst_link.is_symlink() or dst_link.exists():
        dst_link.unlink()

    # 2. 首选主路径：直接从 HuggingFace 官方源下载至本地 NVMe (速度快且杜绝 FUSE 慢读取)
    print(f"   ⬇️ 首选主路径: 从 HuggingFace 官方拉取至本地 NVMe: {fname} ({exact_bytes / (1024**3):.2f} GiB)...")
    download_success = False
    try:
        downloaded = hf_hub_download(
            repo_id="Comfy-Org/MiniMax-H3",
            filename=rel_path,
            local_dir=str(LOCAL_MODELS),
            local_dir_use_symlinks=False
        )
        dl_path = Path(downloaded)
        if dl_path.resolve() != dst_link.resolve() and dl_path.exists():
            shutil.move(str(dl_path), str(dst_link))

        if check_safetensors_file(dst_link, exact_bytes):
            print(f"   ✅ HuggingFace 直连下载完成并通过校验: {fname}")
            download_success = True

            # 3. 同步原子备份至 Drive 容灾缓存（不阻断主流程）
            try:
                drive_sub_file.parent.mkdir(parents=True, exist_ok=True)
                if not drive_sub_file.exists() or not check_safetensors_file(drive_sub_file, exact_bytes):
                    tmp_drive = drive_sub_file.with_suffix(".safetensors.partial")
                    shutil.copyfile(dst_link, tmp_drive)
                    tmp_drive.replace(drive_sub_file)
                    print(f"   💾 已同步原子备份至 Drive 容灾缓存: {drive_sub_file.name}")
            except Exception as be:
                print(f"   ⚠️ 备份到 Drive 略过: {be}")
        else:
            print(f"   ⚠️ HuggingFace 下载文件完整性未通过: {fname}")
            if dst_link.exists():
                dst_link.unlink()
    except Exception as dl_err:
        print(f"   ⚠️ HuggingFace 直连下载受阻: {dl_err}")
        if dst_link.is_symlink() or dst_link.exists():
            try:
                dst_link.unlink()
            except Exception:
                pass

    # 4. 容灾兜底回退：仅当 HuggingFace 直连下载失败时，尝试从 Google Drive 缓存恢复
    if not download_success:
        print(f"   🔄 触发容灾回退: 尝试从 Google Drive 检索可用备份: {fname} ...")
        cand_drive = None
        for cand in [drive_sub_file, drive_flat_file]:
            if cand.exists():
                if check_safetensors_file(cand, exact_bytes):
                    cand_drive = cand
                    break
                else:
                    print(f"   🗑️ 检测到 Drive 中存在残损半截文件: {cand.name}，正在清理...")
                    try:
                        cand.unlink()
                    except Exception:
                        pass

        if cand_drive:
            print(f"   ⚡ 命中 Drive 缓存: {cand_drive.name}")
            copied = False
            try:
                print(f"   📦 正在从 Drive 物理拷贝至本地 NVMe (避免 FUSE 显存饥饿延迟)...")
                shutil.copy2(cand_drive, dst_link)
                if check_safetensors_file(dst_link, exact_bytes):
                    copied = True
                    print(f"   ✅ 从 Drive 物理拷贝完成: {fname}")
            except Exception as ce:
                print(f"   ⚠️ Drive 物理拷贝失败 ({ce})，降级为软链接...")
                if dst_link.exists():
                    dst_link.unlink()

            if not copied:
                dst_link.symlink_to(cand_drive.resolve())
                print(f"   🔗 软链接建立: {fname} -> {cand_drive}")
        else:
            raise RuntimeError(f"❌ HuggingFace 下载失败且 Google Drive 无有效备份: {fname}！请检查网络连接。")

# 5. 建立向后兼容别名软链接 (确保旧版工作流完全兼容)
aliases = [
    (LOCAL_MODELS / "vae" / "minimax_h3_vae.safetensors", LOCAL_MODELS / "vae" / "minimax_h3_video_vae_int8_convrot.safetensors"),
    (LOCAL_MODELS / "loras" / "minimax_h3_fl2va_pdd_8step_pruned_comfy.safetensors", LOCAL_MODELS / "loras" / "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"),
]
for alias_path, src_path in aliases:
    if src_path.exists() and not alias_path.exists():
        alias_path.symlink_to(src_path.resolve())
        print(f"🔗 建立兼容别名: {alias_path.name} -> {src_path.name}")

print(f"\n🎉 全部 5 大核心模型权重秒级就绪！总耗时: {time.time() - t_m0:.2f} 秒")


🔗 正在挂载并校验 5 大核心模型权重 (本地 NVMe 优先 + HF 直连 + Drive 容灾)...
   ⬇️ 首选主路径: 从 HuggingFace 官方拉取至本地 NVMe: minimax_h3_fl2va_pruned_int8_convrot.safetensors (19.53 GiB)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


diffusion_models/minimax_h3_fl2va_pruned(…): reconstructing file:   0%|          |  0.00B / 21.0GB            

diffusion_models/minimax_h3_fl2va_pruned(…): downloading bytes:           |  0.00B            

   ✅ HuggingFace 直连下载完成并通过校验: minimax_h3_fl2va_pruned_int8_convrot.safetensors
   ⬇️ 首选主路径: 从 HuggingFace 官方拉取至本地 NVMe: qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors (14.61 GiB)...


text_encoders/qwen3vl_32b_minimax_h3_nvf(…): reconstructing file:   0%|          |  0.00B / 15.7GB            

... (7 outputs left out)

   ✅ HuggingFace 直连下载完成并通过校验: minimax_h3_audio_vae_fp32.safetensors
   ⬇️ 首选主路径: 从 HuggingFace 官方拉取至本地 NVMe: minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors (1.82 GiB)...


loras/minimax_h3_fl2v_turbo_8step_v1.0_c(…): reconstructing file:   0%|          |  0.00B / 1.96GB            

loras/minimax_h3_fl2v_turbo_8step_v1.0_c(…): downloading bytes:           |  0.00B            

   ✅ HuggingFace 直连下载完成并通过校验: minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors
🔗 建立兼容别名: minimax_h3_vae.safetensors -> minimax_h3_video_vae_int8_convrot.safetensors
🔗 建立兼容别名: minimax_h3_fl2va_pdd_8step_pruned_comfy.safetensors -> minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors

🎉 全部 5 大核心模型权重秒级就绪！总耗时: 135.89 秒


### 步骤 5：启动 ComfyUI 守护服务（防死锁日志重定向与就绪轮询）
- 严禁使用 `subprocess.PIPE` 避免缓冲区满死锁；将日志直接输出至 `/content/comfyui.log`。
- 轮询 120 秒检测服务状态，若提前退出自动打印末尾日志进行诊断。


In [5]:
import subprocess, time, urllib.request, os

%cd /content/ComfyUI
# 启动 ComfyUI 守护服务 (--lowvram 保证 L4 24GB 稳定卸载)
# 必须将 stdout/stderr 重定向到文件，严禁使用 subprocess.PIPE 避免缓冲区死锁
log_file = open('/content/comfyui.log', 'w', encoding='utf-8')
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"🚀 ComfyUI 后台服务已拉起 (PID: {comfy_proc.pid})，日志输出至 /content/comfyui.log")

# 轮询本地接口就绪状态 (最多等待 120s)
ready = False
for i in range(60):
    if comfy_proc.poll() is not None:
        print(f"\n❌ ComfyUI 进程提前异常退出，退出码: {comfy_proc.returncode}")
        break
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        with urllib.request.urlopen(req, timeout=2) as r:
            if r.status == 200:
                print("✅ ComfyUI 本地 API 就绪 (127.0.0.1:8188)！")
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    log_file.flush()
    tail = ""
    if os.path.exists('/content/comfyui.log'):
        with open('/content/comfyui.log', 'r', encoding='utf-8', errors='replace') as f:
            tail = "".join(f.readlines()[-50:])
    print("\n" + "=" * 65)
    print(f"❌ ComfyUI 启动失败或超时，诊断日志如下:\n{tail}")
    print("=" * 65)
    raise RuntimeError("ComfyUI 未能在 120 秒内就绪，请核对上方日志。")


/content/ComfyUI
🚀 ComfyUI 后台服务已拉起 (PID: 2293)，日志输出至 /content/comfyui.log
✅ ComfyUI 本地 API 就绪 (127.0.0.1:8188)！


### 步骤 6：实测 Benchmark 验证（真实冒烟测试：测算单帧耗时与 CU 消耗率）
- 真正向 ComfyUI `/prompt` 提交 1 条短镜头测试任务（960x544, 61帧, 8步），轮询直到出片。
- **跨进程全局显存采样**：通过 CUDA 驱动级接口 `torch.cuda.mem_get_info()` 与 ComfyUI `/system_stats` 实时跟踪整卡显存峰值；若遇超时自动下发 `/interrupt` 释放 GPU 显存。


In [6]:
import json, os, shutil, time, urllib.error, urllib.request
from pathlib import Path
import torch

print("=" * 65)
print("🔍 MiniMax H3 真实冒烟基准评测 (Smoke Benchmark)")
print("=" * 65)

# 官方标准完整签名工作流 (960x544, 124帧, 8步 Turbo)
smoke_wf = {
    "1": {
        "inputs": {
            "unet_name": "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
            "weight_dtype": "default",
        },
        "class_type": "UNETLoader",
    },
    "2": {
        "inputs": {
            "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
            "type": "minimax",
        },
        "class_type": "CLIPLoader",
    },
    "4": {
        "inputs": {
            "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"
        },
        "class_type": "VAELoader",
    },
    "10": {
        "inputs": {
            "lora_name": (
                "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"
            ),
            "strength_model": 1.0,
            "model": ["1", 0],
        },
        "class_type": "LoraLoaderModelOnly",
    },
    "5": {
        "inputs": {"width": 960, "height": 544, "length": 124},
        "class_type": "EmptyMiniMaxH3LatentAV",
        "_meta": {
            "title": "Empty MiniMax H3 Latent"
        },
    },
    "6": {
        "inputs": {
            "text": (
                "A cinematic macro shot of an espresso machine brewing rich"
                " coffee into a clear glass. Slow motion droplets forming with"
                " golden crema. Ambient steam hissing, smooth mechanical pump"
                " humming, gentle liquid dripping acoustic sound."
            ),
            "clip": ["2", 0],
        },
        "class_type": "CLIPTextEncode",
        "_meta": {
            "title": "Positive Prompt"
        },
    },
    "7": {
        "inputs": {
            "text": (
                "distorted audio, watermark, text, blur, low quality, deformed,"
                " jerky motion, noisy static"
            ),
            "clip": ["2", 0],
        },
        "class_type": "CLIPTextEncode",
        "_meta": {
            "title": "Negative Prompt"
        },
    },
    "3": {
        "inputs": {
            "seed": 42,
            "steps": 8,
            "cfg": 3.0,
            "sampler_name": "euler",
            "scheduler": "simple",
            "denoise": 1.0,
            "model": ["10", 0],
            "positive": ["6", 0],
            "negative": ["7", 0],
            "latent_image": ["5", 0],
        },
        "class_type": "KSampler",
        "_meta": {
            "title": "KSampler"
        },
    },
    "8": {
        "inputs": {"samples": ["3", 0], "vae": ["4", 0]},
        "class_type": "VAEDecode",
    },
    "9": {
        "inputs": {
            "filename_prefix": "benchmark_smoke_42",
            "fps": 24.0,
            "lossless": False,
            "quality": 85,
            "method": "default",
            "images": ["8", 0],
        },
        "class_type": "SaveAnimatedWEBP",
        "_meta": {
            "title": "Save Production Video"
        },
    },
}

# 备份冒烟工作流记录（保留生产工作流模板不受冒烟提示词覆盖）
wf_dir = Path("/content/minimax-h3-colab-workflow/workflows")
wf_dir.mkdir(parents=True, exist_ok=True)
with open(
    wf_dir / "h3_t2v_smoke_benchmark.json", "w", encoding="utf-8"
) as f:
    json.dump(smoke_wf, f, indent=2, ensure_ascii=False)

# 2. 提交任务至 ComfyUI
print("🚀 提交冒烟测试任务至 ComfyUI API (960x544, 124 帧, 8 步 Turbo)...")
t_start = time.time()
prompt_data = json.dumps({"prompt": smoke_wf}).encode("utf-8")
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=prompt_data,
    headers={"Content-Type": "application/json"},
)

try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        res_json = json.loads(resp.read().decode("utf-8"))
        prompt_id = res_json.get("prompt_id")
        print(f"✅ 冒烟任务提交成功，Prompt ID: {prompt_id}")
except urllib.error.HTTPError as e:
    err_body = e.read().decode("utf-8", errors="replace")
    print(f"\n❌ ComfyUI 返回 HTTP {e.code} 校验失败，详细报错:\n{err_body}")
    raise RuntimeError(f"ComfyUI 校验未通过: HTTP {e.code}")

# 3. 轮询并实时跟踪显存与去噪耗时
history_data = None
last_log = 0
peak_used_gb = 0.0

while time.time() - t_start < 600:
    elapsed = int(time.time() - t_start)

    # 采样整卡真实显存
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        used_gb = (total_bytes - free_bytes) / (1024**3)
        if used_gb > peak_used_gb:
            peak_used_gb = used_gb

    try:
        hreq = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        with urllib.request.urlopen(hreq, timeout=5) as r:
            hjson = json.loads(r.read().decode("utf-8"))
            if prompt_id in hjson:
                history_data = hjson[prompt_id]
                break
    except Exception:
        pass

    if elapsed - last_log >= 15:
        print(
            f"   ⏳ 渲染去噪中... 已耗时 {elapsed}s | 显存峰值: {peak_used_gb:.2f} GB"
        )
        last_log = elapsed
    time.sleep(3)

total_cost = round(time.time() - t_start, 2)
if not history_data:
    raise TimeoutError("冒烟任务执行超时 (600s)")

# 4. 提取输出与生成报告
out_dir = Path("/content/drive/MyDrive/MiniMax_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)
comfy_out = Path("/content/ComfyUI/output")

found_files = list(comfy_out.glob("benchmark_smoke_42*"))
saved_files = []
for f in found_files:
    dst = out_dir / f.name
    shutil.copy2(f, dst)
    saved_files.append(str(dst))
    print(f"💾 成片已同步至 Google Drive: {dst.name}")

print("\n" + "=" * 65)
print("🎉 MiniMax H3 冒烟基准评测圆满通过！核心数据回收如下：")
print("=" * 65)
print(f"⏱️  单镜头端到端耗时: {total_cost} 秒")
print(
    f"🚀 单步去噪估算: ~{round((total_cost - 30) / 8, 2)} s/it (去噪总步数: 8)"
)
print(
    f"📊 L4 显存实际峰值: {peak_used_gb:.2f} GB / 22.03 GB (安全余量: {22.03 - peak_used_gb:.2f} GB)"
)
print(f"💰 单镜头消耗算力估算: ~{round((total_cost / 3600) * 3.5, 3)} CU")
print(f"📂 成果已保存至: {out_dir.resolve()}")
print("=" * 65)


🔍 MiniMax H3 真实冒烟基准评测 (Smoke Benchmark)
🚀 提交冒烟测试任务至 ComfyUI API (960x544, 124 帧, 8 步 Turbo)...
✅ 冒烟任务提交成功，Prompt ID: b1cea508-89d3-4f8b-8df3-d793c797e1a1
   ⏳ 渲染去噪中... 已耗时 15s | 显存峰值: 13.99 GB
   ⏳ 渲染去噪中... 已耗时 30s | 显存峰值: 13.99 GB
   ⏳ 渲染去噪中... 已耗时 45s | 显存峰值: 13.99 GB
   ⏳ 渲染去噪中... 已耗时 60s | 显存峰值: 16.26 GB
   ⏳ 渲染去噪中... 已耗时 75s | 显存峰值: 19.74 GB
   ⏳ 渲染去噪中... 已耗时 90s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 105s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 120s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 135s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 150s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 165s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 180s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 195s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 210s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 225s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 240s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 255s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 270s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 285s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 300s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中... 已耗时 315s | 显存峰值: 21.77 GB
   ⏳ 渲染去噪中..

### 步骤 7：无头批量生产队列（自动出片 + 自动同步 Drive）
- 支持音画同频描述，成片与结构化日志自动保存至 `/content/drive/MyDrive/MiniMax_Outputs`。


In [7]:
import os, sys
from pathlib import Path

repo_path = Path("/content/minimax-h3-colab-workflow")
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from scripts.batch_runner import run_batch

run_batch(
    workflow_path=str(repo_path / "workflows" / "h3_t2v_pdd8_production.json"),
    prompts_path=str(repo_path / "configs" / "prompts.example.json"),
    output_dir="/content/drive/MyDrive/MiniMax_Outputs",
    server_url="http://127.0.0.1:8188",
    comfy_output_dir="/content/ComfyUI/output",
    timeout_per_shot=900
)


🎬 MiniMax H3 批处理队列执行器 (Batch Runner 2.0 - BasicGuider / No-CFG)
📄 工作流模板: /content/minimax-h3-colab-workflow/workflows/h3_t2v_pdd8_production.json
📋 任务清单: /content/minimax-h3-colab-workflow/configs/prompts.example.json
🎯 动态绑定的活动权重: {'diffusion_model': 'minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'video_vae': 'minimax_h3_video_vae_int8_convrot.safetensors', 'lora': 'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'}
💾 最终输出落盘路径: /content/drive/MyDrive/MiniMax_Outputs
📊 共检测到 3 个排产镜头任务

[1/3] 正在启动: shot_01_coffee_macro - 咖啡机微距流速
   📐 规格: 960x544 | 124 帧 | 步数 8 | 种子 42
   📝 提示词: A close-up macro shot of dark roast espresso dripping slowly from a bottomless portafilter...
   📐 [节点 5: EmptyMiniMaxH3LatentAV] 规格尺寸注入: 960x544, 124 帧
   📝 [节点 6] 正向提示词注入成功: A close-up macro shot of dark roast espresso dripp...
   🛡️ [节点 7] 负向提示词注入成功: watermark, text, blur, low quality, defo...
   🚀 已推入执行队列 (Prompt ID: 67231c95-5a1f-4087-af92-628b9ce71b24)
   ⏳ 渲染去噪中... 已耗时 21s
   ⏳ 渲染去噪中... 已耗时 42s

### 步骤 8：防扣费与安全收尾（跑完务必执行）
> ⚠️ **重要**：运行下方单元格立即刷盘并销毁实例，彻底切断算力扣费！


In [8]:
import os
from google.colab import runtime

print("🛑 正在执行安全收尾：刷盘并销毁 Colab 运行时，彻底终止算力计费...")
try:
    os.sync()
    runtime.unassign()
    print("✅ runtime.unassign() 已成功下发！")
except Exception as e:
    print(f"⚠️ runtime.unassign() 异常: {e}，触发容器级兜底强制断开...")
    try:
        os.sync()
        os.kill(1, 15)
    except Exception:
        os.system("kill -9 1")


🛑 正在执行安全收尾：刷盘并销毁 Colab 运行时，彻底终止算力计费...
✅ runtime.unassign() 已成功下发！
